# Center-grating analysis

This notebook analyzes `spotWithAnnularGrating` with the grating restricted to the
**center**. It follows the same workflow as the cone center/annulus notebooks:
database refresh, condition discovery, one-cell analysis, persistent saving,
population analysis, and an example-stimulus visualization.

The light level is reconstructed per epoch block. Fixed `EL...` filters come from
the raw Stage device configurator; embedded `FW...` text there is ignored. The
numeric FilterWheel reading comes independently from protected metadata and must
agree across every epoch in a block. The resulting maximum is the calibrated R*/s
at normalized display intensity 1; the actual background is
`max_light_level × backgroundIntensity`.


In [6]:
import sys
import time

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        f'This notebook requires the retinanalysis Python 3.11 kernel; '
        f'got Python {sys.version.split()[0]} at {sys.executable}')

import_started = time.perf_counter()
import numpy as np
import pandas as pd
from IPython.display import display

import retinanalysis as ra
from retinanalysis.SCutils import explore as sc
from retinanalysis.SCutils.protocols import spot_annular_grating as sag

SITE = 'center'
SITE_LABEL = 'Center-grating analysis'
MAX_SERIES_RESISTANCE = 30e6
STORE_PATH = sag.store_dir() / f'{SITE}_grating'

print(f'Python {sys.version.split()[0]} | {sys.executable}')
print(f'Imports ready in {time.perf_counter() - import_started:.2f} s')
print(f'Saved records: {STORE_PATH}')

Python 3.11.13 | /Users/chrischen/opt/anaconda3/envs/retinanalysis/bin/python
Imports ready in 0.00 s
Saved records: /Volumes/ChrisNewSSD/retinanalysis_output/spot_annular_grating/center_grating


## 1. Load or refresh the single-cell database

Set `UPDATE_DATABASE = True` when new Symphony H5/JSON metadata should be ingested.
The default connects to the existing local DataJoint database without changing it.


In [7]:
UPDATE_DATABASE = False
ra.djconnect()

if UPDATE_DATABASE:
    database_report = ra.populate_database()
    print(f"newly added: {len(database_report['added'])}")
    print(f"refreshed: {len(database_report['updated'])}")
    print(f"errored: {len(database_report['skipped'])}")
    print(f"database: {len(database_report['experiments'])} experiments; "
          f"{len(database_report['stale'])} still out of date")
else:
    print('Connected to the existing database. Set UPDATE_DATABASE=True to ingest changes.')


Connected to the existing database. Set UPDATE_DATABASE=True to ingest changes.


## 2. Search the database for center-grating recordings

Discovery keeps every recorded cell type, numeric FilterWheel setting, bright-bar
contrast, and bar width. Series resistance still resolves and validates
`onlineAnalysis`. Each row is one explicit cell × mode × NDF combination ×
background × bright contrast × bar-width condition, although the compact table
omits the bright-contrast and database bookkeeping columns requested for this view.

`ndf_combination` and `max_light_level` come from the block-level light reader; no
protocol `maxIntensity` parameter is assumed. `date_index` uses the complete sorted
protocol date list, so the same value works in Section 3 even when a date has no
recording for this grating site.

In [ ]:
protocol_index = sc.find_blocks(sag.PROTOCOL, show=False)
protocol_dates = sorted(protocol_index.exp_name.dropna().unique())
date_index_map = {exp_name: index + 1 for index, exp_name in enumerate(protocol_dates)}

df_blocks = sag.find_blocks(show=False)
df_blocks = sag.check_series_resistance(
    df_blocks, max_series_resistance=MAX_SERIES_RESISTANCE)
site_blocks = df_blocks[df_blocks.grating_site.eq(SITE)].copy()

selected = sag.group_blocks(
    site_blocks, show=False,
    require_filter_wheel=False,
    allowed_bright_contrast=None,
    min_bar_width=None,
    min_epochs=None,
    separate_bright_contrast=True,
    collapse_bar_widths=False)
selected = selected.sort_values(
    ['exp_name', 'cell_label', 'onlineAnalysis', 'ndf_combination',
     'backgroundIntensity', 'bright', 'bar_width']).reset_index(drop=True)
selected.insert(0, 'date_index', selected.exp_name.map(date_index_map).astype(int))

print(f'{SITE_LABEL}: {len(selected)} conditions across '
      f'{selected.exp_name.nunique()} experiments and '
      f"{selected.groupby(['exp_name', 'cell_label']).ngroups} cells")
condition_columns = [
    'date_index', 'exp_name', 'cell_label', 'cell_type_short', 'onlineAnalysis',
    'ndf_combination', 'filter_wheel_ndf', 'max_light_level',
    'backgroundIntensity', 'spot_intensity', 'bar_width', 'aperture',
    'annulus_inner', 'annulus_outer',
]
condition_columns = [column for column in condition_columns if column in selected]
sc.scroll_table(
    selected[condition_columns], height=430,
    num_cols=('date_index', 'filter_wheel_ndf', 'max_light_level',
              'backgroundIntensity', 'spot_intensity', 'bar_width', 'aperture',
              'annulus_inner', 'annulus_outer'))

  2026-01-02_E_2: cannot open the h5 (FileNotFoundError) — 2 block(s) have no series-resistance reading


## 3. Analyze one cell across its recorded conditions

Enter a `date_index`, cell label, and resolved `onlineAnalysis`; no background or
NDF selection is required. This standalone section finds every matching condition
and prints a condition table when there is more than one. Each unique fixed-NDF +
FilterWheel combination, background intensity, and bright-bar contrast is analyzed
separately.

If more than one bright-bar contrast or bar width was recorded, an alert is printed. With
`COLLAPSE_BAR_WIDTHS = False` (the default), each width is also analyzed and saved as
a separate condition. Set it to `True` only when you deliberately want to pool all
bar widths within each otherwise-identical condition.

When multiple fixed-NDF + FilterWheel + background combinations are present,
their tuning curves are overlaid after the individual condition plots. The overlay
labels retain bright-bar contrast and bar width so any additional stimulus changes
remain visible.

Before traces are loaded, each condition prints cell identity and time, stimulus
parameters, light calibration, block IDs, and epoch count. Series resistance remains
part of the analysis/QC even though it is omitted from the compact Section 2 table.

In [ ]:
# Standalone after the import cell; Section 2 is optional.
DATE_INDEX = 27
CELL_LABEL = 'Cell4'
ONLINE_ANALYSIS = 'exc'  # 'extracellular', 'exc', or 'inh'
COLLAPSE_BAR_WIDTHS = False

protocol_index = sc.find_blocks(sag.PROTOCOL, show=False)
protocol_dates = sorted(protocol_index.exp_name.dropna().unique())
if not 1 <= DATE_INDEX <= len(protocol_dates):
    raise ValueError(f'date_index {DATE_INDEX} is outside 1-{len(protocol_dates)}')
EXP_NAME = protocol_dates[DATE_INDEX - 1]

date_blocks = sag.find_blocks(exp_names=[EXP_NAME], show=False)
date_blocks = sag.check_series_resistance(
    date_blocks, max_series_resistance=MAX_SERIES_RESISTANCE, show=False)
date_blocks = date_blocks[date_blocks.grating_site.eq(SITE)].copy()

cell_blocks = date_blocks[
    date_blocks.cell_label.eq(CELL_LABEL)
    & date_blocks.onlineAnalysis.eq(ONLINE_ANALYSIS)
].copy()
if cell_blocks.empty:
    available = (date_blocks[
        ['cell_label', 'cell_type_short', 'onlineAnalysis']]
        .drop_duplicates().sort_values(['cell_label', 'onlineAnalysis']))
    display(available)
    raise ValueError(
        f'No {SITE}-grating blocks for {EXP_NAME} {CELL_LABEL} {ONLINE_ANALYSIS}')

bright_contrast_values = sorted(cell_blocks.bright.dropna().astype(float).unique())
if len(bright_contrast_values) > 1:
    print(f'ALERT: multiple bright-bar contrasts were recorded: {bright_contrast_values}.')

bar_width_values = sorted(cell_blocks.bar_width.dropna().astype(float).unique())
if len(bar_width_values) > 1:
    action = 'COLLAPSING them by request' if COLLAPSE_BAR_WIDTHS else 'keeping them separate'
    print(f'ALERT: multiple bar widths were recorded: {bar_width_values} µm; {action}.')

condition_rows = sag.group_blocks(
    cell_blocks, show=False,
    require_filter_wheel=False,
    allowed_bright_contrast=None,
    min_bar_width=None,
    min_epochs=None,
    separate_bright_contrast=True,
    collapse_bar_widths=COLLAPSE_BAR_WIDTHS)
if condition_rows.empty:
    raise ValueError(f'No conditions found for {EXP_NAME} {CELL_LABEL}')

condition_rows = condition_rows.sort_values(
    ['ndf_combination', 'backgroundIntensity', 'bright', 'bar_width'],
    na_position='last').reset_index(drop=True)
condition_rows.insert(0, 'condition_index', np.arange(1, len(condition_rows) + 1))

condition_view_columns = [
    'condition_index', 'ndf_combination', 'filter_wheel_ndf',
    'backgroundIntensity', 'bright', 'bar_width', 'max_light_level', 'rstar',
    'blocks', 'epochs', 'block_ids',
]
if len(condition_rows) > 1:
    print(f'ALERT: {len(condition_rows)} unique conditions found; each will be analyzed separately:')
    display(condition_rows[condition_view_columns])

light_conditions = condition_rows[
    ['ndf_combination', 'filter_wheel_ndf', 'backgroundIntensity']].drop_duplicates()
if len(light_conditions) > 1:
    print(f'ALERT: {len(light_conditions)} NDF/FilterWheel/background combinations found; '
          'each becomes a separate saved entry and their tuning curves will be overlaid.')

records = []
condition_figures = []
for _, condition_row in condition_rows.iterrows():
    condition_block_ids = [int(value) for value in condition_row.block_ids.split(',')]
    condition_blocks = date_blocks[date_blocks.block_id.isin(condition_block_ids)].copy()
    metadata = pd.DataFrame([{
        'condition_index': int(condition_row.condition_index),
        'cell_label': condition_row.cell_label,
        'cell_type': condition_row.cell_type_short,
        'onlineAnalysis': condition_row.onlineAnalysis,
        'spotIntensity': condition_row.spot_intensity,
        'brightBarContrast': condition_row.bright,
        'barWidth_um': condition_row.bar_width,
        'apertureDiameter_um': condition_row.aperture,
        'annulusInnerDiameter_um': condition_row.annulus_inner,
        'annulusOuterDiameter_um': condition_row.annulus_outer,
        'background_Rstar_per_s': condition_row.rstar,
    }])
    print(f'Condition {int(condition_row.condition_index)}/{len(condition_rows)} metadata:')
    display(metadata.T.rename(columns={0: 'value'}))

    condition_record = sag.analyze_group(
        EXP_NAME, condition_block_ids,
        online_analysis=ONLINE_ANALYSIS,
        max_series_resistance=MAX_SERIES_RESISTANCE,
        keep_raw=True)
    records.append(condition_record)
    condition_figures.append(sag.plot_group(condition_record))

light_tuning_figure = None
if len(light_conditions) > 1:
    overlay_labels = [
        (f'C{int(row.condition_index)}: NDF={row.ndf_combination}, '
         f'FW={row.filter_wheel_ndf}, background={row.backgroundIntensity}, '
         f'bright={row.bright}, width={row.bar_width} µm')
        for _, row in condition_rows.iterrows()
    ]
    print('Overlaying tuning curves across the recorded light conditions.')
    light_tuning_figure = sag.plot_tuning_overlay(
        records, labels=overlay_labels,
        title=f'{EXP_NAME} {CELL_LABEL}: tuning curves by light condition')

print(f'Analyzed {len(records)} separate condition(s) for {EXP_NAME} {CELL_LABEL}.')

AttributeError: 'DataFrame' object has no attribute 'bright'

### 3a. Save these conditions for population analysis

Save every analyzed condition to the site-specific HDF5 store. The key includes
date, cell, mode, site, fixed NDFs, numeric FilterWheel, background, bright-bar
contrast, and the analyzed bar-width set. Therefore multiple light/background/bar
conditions from the same cell remain separate entries; rerunning an identical
condition replaces only that entry.

In [ ]:
if not records:
    raise ValueError('Run Section 3 before saving')
condition_output_path = sag.save_records(records, path=STORE_PATH)
print(f'Saved {len(records)} separate condition(s) to {condition_output_path}')

### 3b. Check saved conditions

Load only the scalar index. Every NDF/background/bright/bar-width condition is one
row; trace arrays remain on disk until a plot requests them.

In [ ]:
saved_cells = sag.load_summary(path=STORE_PATH)
saved_columns = [
    'exp_name', 'cell_label', 'cell_type', 'online_analysis', 'ndf_combination',
    'max_light_level', 'background_intensity', 'bright_bar_contrast',
    'bar_widths', 'rstar', 'series_resistance', 'n_epochs_high_rs',
    'n_epochs', 'block_ids',
]
saved_columns = [column for column in saved_columns if column in saved_cells]
print(f'{len(saved_cells)} saved {SITE}-grating condition(s)')
sc.scroll_table(
    saved_cells[saved_columns], height=320,
    num_cols=('max_light_level', 'background_intensity', 'bright_bar_contrast',
              'rstar', 'series_resistance', 'n_epochs_high_rs', 'n_epochs'))

## 4. Population analysis

Population work includes every saved cell type. Conditions are labelled as
`cell type / grating site`, and recording modes remain separate because firing rate
and current have different units. Bright-bar contrasts are plotted in separate figure
sets so their different cone predictions are never mixed.

In [ ]:
summary = sag.add_condition(sag.load_summary(path=STORE_PATH))
if summary.empty:
    raise ValueError(f'No saved {SITE}-grating records in {STORE_PATH}')

population_table = (summary.groupby(
    ['cell_type_short', 'online_analysis', 'ndf_combination',
     'background_intensity', 'bright_bar_contrast', 'bar_widths', 'rstar_level'],
    dropna=False)
    .agg(recordings=('key', 'size'), cells=('cell_label', 'nunique'),
         epochs=('n_epochs', 'sum'), mean_rstar=('rstar', 'mean'),
         crossing_mean=('crossing_interp', 'mean'),
         crossing_sem=('crossing_interp', 'sem'))
    .reset_index())
sc.scroll_table(
    population_table, height=320,
    num_cols=('background_intensity', 'bright_bar_contrast', 'rstar_level',
              'recordings', 'cells', 'epochs', 'mean_rstar',
              'crossing_mean', 'crossing_sem'))

bright_values = sorted(summary.bright_bar_contrast.dropna().astype(float).unique())
for bright_value in bright_values:
    bright_summary = summary[np.isclose(summary.bright_bar_contrast, bright_value)].copy()
    available_conditions = tuple(bright_summary.condition.dropna().drop_duplicates())
    available_modes = tuple(bright_summary.online_analysis.dropna().drop_duplicates())
    print(f'Population figures: brightBarContrast={bright_value:g}; '
          f'{len(bright_summary)} saved conditions')
    sag.plot_weber_comparison(
        bright_summary, bright_contrast=bright_value,
        conditions=available_conditions, modes=available_modes)
    sag.plot_population_tuning(
        bright_summary, conditions=available_conditions, modes=available_modes,
        allowed_bright_contrast=None)

### 4a. Example saved recording from each cell-type/site and mode

For each bright-bar contrast, show the most deeply sampled saved recording from
every available cell-type/site condition and recording mode.

In [ ]:
all_records = sag.load_records(path=STORE_PATH)
for bright_value in bright_values:
    bright_summary = summary[np.isclose(summary.bright_bar_contrast, bright_value)].copy()
    available_conditions = tuple(bright_summary.condition.dropna().drop_duplicates())
    available_modes = tuple(bright_summary.online_analysis.dropna().drop_duplicates())
    bright_keys = set(bright_summary.key)
    bright_records = {key: value for key, value in all_records.items() if key in bright_keys}
    print(f'Examples: brightBarContrast={bright_value:g}')
    sag.plot_condition_examples(
        bright_records, conditions=available_conditions, modes=available_modes)

## 5. Example center-grating stimulus

Render the selected block using the recorded aperture, annulus, bar width,
background, spot intensity, and bright/dark contrasts. The geometry should visibly
place the grating over the **center**; the center spot remains an independent protocol
parameter for surround recordings.


In [ ]:
EXAMPLE_CONDITION_INDEX = 1
if not 1 <= EXAMPLE_CONDITION_INDEX <= len(records):
    raise ValueError(f'EXAMPLE_CONDITION_INDEX must be 1-{len(records)}')
example_record = records[EXAMPLE_CONDITION_INDEX - 1]
example_block = int(example_record.block_ids[0])
stim = ra.StimBlock(example_record.exp_name, example_block, verbose=False)
example_parameters = stim.df_epochs['epoch_parameters'].iloc[0]
dark_values = np.sort(stim.df_epochs['currentDarkContrast'].dropna().unique())
example_dark = dark_values[[0, len(dark_values) // 2, -1]]
stimulus_figure = sag.plot_stimulus_schematic(
    example_parameters, dark_contrasts=example_dark)